# Demo — Pipeline de Agentes Clínicos (ADK + DeepSeek)

Este notebook roda o mesmo pipeline de `run_demo.py` (`src/pipeline.py::run_case`), célula a
célula, para inspecionar cada etapa (extração, checagem de segurança, relatório) separadamente.

⚠️ Apoio à decisão clínica — não substitui avaliação médica presencial.

In [ ]:
import json
import os
import sys

sys.path.insert(0, os.path.abspath("."))  # garante que `src` seja importável (notebook na raiz do repo)

from dotenv import load_dotenv

load_dotenv()

from src.data_prep import load_sample
from src.pipeline import run_case

In [ ]:
sample = load_sample(os.environ.get("DATA_CSV_PATH", "data/mtsamples.csv"), n_total=20)
sample.head()

## Caso real (amostra do mtsamples)

In [ ]:
case = sample.iloc[0].to_dict()
result = run_case(case)

print("--- Extração ---")
print(json.dumps(result["extracted"], ensure_ascii=False, indent=2))

print("\n--- Segurança ---")
print(json.dumps(result["safety_report"], ensure_ascii=False, indent=2))

print("\n--- RELATÓRIO ---")
print(result["relatorio"])

## Caso sintético — interação perigosa não explícita no texto original

O texto nunca menciona "interação" ou "risco" — apenas relata que a paciente já toma Warfarin
e recebeu uma nova prescrição de Aspirin. O Safety Agent identifica o risco cruzando os dois
fármacos extraídos contra a tabela mockada de interações.

In [ ]:
synthetic_case = {
    "description": "Caso sintético para demonstrar interação medicamentosa não explícita no texto.",
    "medical_specialty": "Cardiovascular / Pulmonary (sintético)",
    "transcription": (
        "SUBJECTIVE: Paciente de 68 anos, em uso contínuo de Warfarin para fibrilação atrial, "
        "relata dor torácica leve nos últimos dois dias. Nega falta de ar. "
        "OBJECTIVE: Sinais vitais estáveis. ECG sem alterações agudas. "
        "PLAN: Iniciado Aspirin 100mg ao dia para controle sintomático da dor torácica."
    ),
}
result = run_case(synthetic_case)

print("--- Segurança ---")
print(json.dumps(result["safety_report"], ensure_ascii=False, indent=2))

print("\n--- RELATÓRIO ---")
print(result["relatorio"])